In [ ]:
# import os
# import glob
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.express as px
# import sys

# root = os.path.abspath('../..')  
# sys.path.append(root)

# def plot_dgh_traces_with_breakpoints(
#     path_dgh,
#     path_sec_files,
#     vp_column='Vertical Position m',
#     sec_column='Corrected sp Cond [µS/cm]',
#     id_column='ID',
#     bp1_column='breakpoint_1 (vp)',
#     bp2_column='breakpoint_2 (vp)',
#     title='Traces with breakpoints'
# ):
#     """
#     Lee un CSV de DGH con IDs y breakpoints, busca los archivos CSV de traces
#     que empiezan con cada ID en otra carpeta, y los grafica de forma interactiva
#     en Plotly con breakpoints superpuestos.

#     Parámetros
#     ----------
#     path_dgh : str
#         Ruta al CSV con columnas ID, breakpoint_1 (vp), breakpoint_2 (vp)
#     path_sec_files : str
#         Carpeta donde están los CSV de traces
#     vp_column : str
#         Nombre de la columna de posición vertical en los traces
#     sec_column : str
#         Nombre de la columna de conductividad corregida en los traces
#     id_column : str
#         Nombre de la columna ID en el CSV DGH
#     bp1_column : str
#         Nombre de la columna del primer breakpoint
#     bp2_column : str
#         Nombre de la columna del segundo breakpoint
#     title : str
#         Título de la figura

#     Retorna
#     -------
#     fig : plotly.graph_objects.Figure
#     """
    
#     # -------------------------
#     # Leer y validar archivo DGH
#     # -------------------------
#     df_dgh = pd.read_csv(path_dgh)
#     required_dgh_cols = [id_column, bp1_column, bp2_column]
#     missing_dgh = [c for c in required_dgh_cols if c not in df_dgh.columns]
#     if missing_dgh:
#         raise ValueError(f"Faltan columnas en path_dgh: {missing_dgh}")

#     df_dgh = df_dgh[required_dgh_cols].copy()
#     df_dgh[id_column] = df_dgh[id_column].astype(str)

#     # -------------------------
#     # Colores por ID
#     # -------------------------
#     unique_ids = df_dgh[id_column].dropna().unique().tolist()
#     palette = px.colors.qualitative.Alphabet
#     color_map = {id_val: palette[i % len(palette)] for i, id_val in enumerate(unique_ids)}

#     fig = go.Figure()
#     shown_legend_ids = set()

#     # -------------------------
#     # Iterar por cada ID
#     # -------------------------
#     for _, row in df_dgh.iterrows():
#         id_val = str(row[id_column])
#         bp_values = [row[bp1_column], row[bp2_column]]
#         color = color_map[id_val]

#         # Buscar archivos que empiecen con el ID
#         pattern = os.path.join(path_sec_files, f"{id_val}*.csv")
#         matched_files = sorted(glob.glob(pattern))

#         if not matched_files:
#             print(f"[AVISO] No se encontraron archivos para ID={id_val} con patrón: {pattern}")
#             continue

#         for file_path in matched_files:
#             try:
#                 df_trace = pd.read_csv(file_path)
#             except Exception as e:
#                 print(f"[AVISO] No se pudo leer {file_path}: {e}")
#                 continue

#             # Validar columnas
#             missing_trace = [c for c in [vp_column, sec_column] if c not in df_trace.columns]
#             if missing_trace:
#                 print(f"[AVISO] En {os.path.basename(file_path)} faltan columnas: {missing_trace}")
#                 continue

#             # Limpiar datos
#             df_trace = df_trace[[vp_column, sec_column]].copy()
#             df_trace[vp_column] = pd.to_numeric(df_trace[vp_column], errors='coerce')
#             df_trace[sec_column] = pd.to_numeric(df_trace[sec_column], errors='coerce')
#             df_trace = df_trace.dropna(subset=[vp_column, sec_column])

#             if df_trace.empty:
#                 print(f"[AVISO] {os.path.basename(file_path)} quedó vacío después de limpiar NaNs.")
#                 continue

#             trace_name = os.path.basename(file_path)

#             # Mostrar leyenda solo una vez por ID en la línea principal
#             showlegend_line = id_val not in shown_legend_ids

#             fig.add_trace(
#                 go.Scatter(
#                     x=df_trace[sec_column],
#                     y=df_trace[vp_column],
#                     mode='markers',
#                     name=f"{id_val}",
#                     legendgroup=id_val,
#                     showlegend=showlegend_line,
#                     line=dict(color=color, width=2),
#                     marker=dict(
#                         color=color,
#                         size=2,
#                         symbol='circle'
#                     ),
#                     hovertemplate=(
#                         f"ID: {id_val}<br>"
#                         f"Archivo: {trace_name}<br>"
#                         f"{sec_column}: %{{x}}<br>"
#                         f"{vp_column}: %{{y}}<extra></extra>"
#                     )
#                 )
#             )

#             shown_legend_ids.add(id_val)

#             # Breakpoints: interpolar x correspondiente al y=breakpoint
#             for i_bp, bp in enumerate(bp_values, start=1):
#                 if pd.isna(bp):
#                     continue

#                 # Buscar el valor de sec_column en el breakpoint usando interpolación
#                 df_sorted = df_trace.sort_values(vp_column).drop_duplicates(subset=[vp_column])

#                 if len(df_sorted) < 2:
#                     continue

#                 y_min = df_sorted[vp_column].min()
#                 y_max = df_sorted[vp_column].max()

#                 if not (y_min <= bp <= y_max):
#                     print(
#                         f"[AVISO] Breakpoint {bp} de ID={id_val} "
#                         f"fuera del rango de {trace_name} ({y_min}, {y_max})"
#                     )
#                     continue

#                 x_bp = None
#                 try:
#                     x_bp = pd.Series(
#                         index=df_sorted[vp_column].values,
#                         data=df_sorted[sec_column].values
#                     ).sort_index().interpolate(method='index').reindex(
#                         sorted(set(df_sorted[vp_column].tolist() + [bp]))
#                     ).interpolate(method='index').loc[bp]
#                 except Exception:
#                     # fallback simple
#                     x_bp = None

#                 if x_bp is None or pd.isna(x_bp):
#                     continue

#                 fig.add_trace(
#                     go.Scatter(
#                         x=[x_bp],
#                         y=[bp],
#                         mode='markers',
#                         name=f"{id_val} - breakpoint {i_bp}",
#                         legendgroup=id_val,
#                         showlegend=False,
#                         marker=dict(
#                             color=color,
#                             size=10,
#                             symbol='x'
#                         ),
#                         hovertemplate=(
#                             f"ID: {id_val}<br>"
#                             f"Archivo: {trace_name}<br>"
#                             f"Breakpoint {i_bp}<br>"
#                             f"{sec_column}: %{{x}}<br>"
#                             f"{vp_column}: %{{y}}<extra></extra>"
#                         )
#                     )
#                 )

#                 # Línea horizontal opcional para visualizar breakpoint
#                 fig.add_hline(
#                     y=bp,
#                     line=dict(color=color, dash='dot', width=1),
#                     opacity=0.45
#                 )

#     fig.update_layout(
#         title=title,
#         xaxis_title=sec_column,
#         yaxis_title=vp_column,
#         template='plotly_white',
#         legend_title='ID',
#         hovermode='closest',
#         height=600
#     )

#     fig.update_yaxes(autorange='reversed')

#     return fig

In [ ]:
# import os
# import glob
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.express as px
# import sys

# root = os.path.abspath('../..')
# sys.path.append(root)


# def plot_dgh_traces_with_breakpoints(
#     path_dgh,
#     path_sec_files,
#     vp_column='Vertical Position m',
#     sec_column='Corrected sp Cond [µS/cm]',
#     id_column='ID',
#     bp1_sec_column='breakpoint_1 (sec)',
#     bp2_sec_column='breakpoint_2 (sec)',
#     bp1_vp_column='breakpoint_1 (vp)',
#     bp2_vp_column='breakpoint_2 (vp)',
#     title='Traces with breakpoints'
# ):
#     """
#     Lee un CSV de DGH con IDs y breakpoints (sec, vp), busca los archivos CSV
#     de traces que empiezan con cada ID en otra carpeta, y los grafica en Plotly
#     con los breakpoints superpuestos SIN interpolación.

#     Parámetros
#     ----------
#     path_dgh : str
#         Ruta al CSV con columnas:
#         ID, breakpoint_1 (sec), breakpoint_2 (sec),
#         breakpoint_1 (vp), breakpoint_2 (vp)
#     path_sec_files : str
#         Carpeta donde están los CSV de traces
#     vp_column : str
#         Nombre de la columna de posición vertical en los traces
#     sec_column : str
#         Nombre de la columna de conductividad corregida en los traces
#     id_column : str
#         Nombre de la columna ID en el CSV DGH
#     bp1_sec_column, bp2_sec_column : str
#         Nombre de las columnas de conductividad de los breakpoints
#     bp1_vp_column, bp2_vp_column : str
#         Nombre de las columnas de posición vertical de los breakpoints
#     title : str
#         Título de la figura

#     Retorna
#     -------
#     fig : plotly.graph_objects.Figure
#     """

#     # -------------------------
#     # Leer y validar archivo DGH
#     # -------------------------
#     df_dgh = pd.read_csv(path_dgh)

#     required_dgh_cols = [
#         id_column,
#         bp1_sec_column, bp2_sec_column,
#         bp1_vp_column, bp2_vp_column
#     ]
#     missing_dgh = [c for c in required_dgh_cols if c not in df_dgh.columns]
#     if missing_dgh:
#         raise ValueError(f"Faltan columnas en path_dgh: {missing_dgh}")

#     df_dgh = df_dgh[required_dgh_cols].copy()
#     df_dgh[id_column] = df_dgh[id_column].astype(str)

#     # Convertir breakpoints a numérico
#     for col in [bp1_sec_column, bp2_sec_column, bp1_vp_column, bp2_vp_column]:
#         df_dgh[col] = pd.to_numeric(df_dgh[col], errors='coerce')

#     # -------------------------
#     # Colores por ID
#     # -------------------------
#     unique_ids = df_dgh[id_column].dropna().unique().tolist()
#     palette = px.colors.qualitative.Alphabet
#     color_map = {id_val: palette[i % len(palette)] for i, id_val in enumerate(unique_ids)}

#     fig = go.Figure()
#     shown_legend_ids = set()

#     # -------------------------
#     # Iterar por cada ID
#     # -------------------------
#     for _, row in df_dgh.iterrows():
#         id_val = str(row[id_column])
#         color = color_map[id_val]

#         breakpoints = [
#             (row[bp1_sec_column], row[bp1_vp_column], 1),
#             (row[bp2_sec_column], row[bp2_vp_column], 2),
#         ]

#         # Buscar archivos que empiecen con el ID
#         pattern = os.path.join(path_sec_files, f"{id_val}*.csv")
#         matched_files = sorted(glob.glob(pattern))

#         if not matched_files:
#             print(f"[AVISO] No se encontraron archivos para ID={id_val} con patrón: {pattern}")
#             continue

#         for file_path in matched_files:
#             try:
#                 df_trace = pd.read_csv(file_path)
#             except Exception as e:
#                 print(f"[AVISO] No se pudo leer {file_path}: {e}")
#                 continue

#             # Validar columnas
#             missing_trace = [c for c in [vp_column, sec_column] if c not in df_trace.columns]
#             if missing_trace:
#                 print(f"[AVISO] En {os.path.basename(file_path)} faltan columnas: {missing_trace}")
#                 continue

#             # Limpiar datos del trace
#             df_trace = df_trace[[vp_column, sec_column]].copy()
#             df_trace[vp_column] = pd.to_numeric(df_trace[vp_column], errors='coerce')
#             df_trace[sec_column] = pd.to_numeric(df_trace[sec_column], errors='coerce')
#             df_trace = df_trace.dropna(subset=[vp_column, sec_column])

#             if df_trace.empty:
#                 print(f"[AVISO] {os.path.basename(file_path)} quedó vacío después de limpiar NaNs.")
#                 continue

#             trace_name = os.path.basename(file_path)

#             # Mostrar leyenda solo una vez por ID
#             showlegend_line = id_val not in shown_legend_ids

#             # Trace principal: puntos reales del CSV, sin líneas, sin interpolación
#             fig.add_trace(
#                 go.Scatter(
#                     x=df_trace[sec_column],
#                     y=df_trace[vp_column],
#                     mode='markers',
#                     name=f"{id_val}",
#                     legendgroup=id_val,
#                     showlegend=showlegend_line,
#                     marker=dict(
#                         color=color,
#                         size=4,
#                         symbol='circle'
#                     ),
#                     hovertemplate=(
#                         f"ID: {id_val}<br>"
#                         f"Archivo: {trace_name}<br>"
#                         f"{sec_column}: %{{x}}<br>"
#                         f"{vp_column}: %{{y}}<extra></extra>"
#                     )
#                 )
#             )

#             shown_legend_ids.add(id_val)

#             # -------------------------
#             # Breakpoints: usar coordenadas directas del CSV DGH
#             # SIN interpolación
#             # -------------------------
#             for bp_sec, bp_vp, i_bp in breakpoints:
#                 if pd.isna(bp_sec) or pd.isna(bp_vp):
#                     continue

#                 fig.add_trace(
#                     go.Scatter(
#                         x=[bp_sec],
#                         y=[bp_vp],
#                         mode='markers',
#                         name=f"{id_val} - breakpoint {i_bp}",
#                         legendgroup=id_val,
#                         showlegend=False,
#                         marker=dict(
#                             color=color,
#                             size=11,
#                             symbol='x',
#                             line=dict(width=1)
#                         ),
#                         hovertemplate=(
#                             f"ID: {id_val}<br>"
#                             f"Archivo: {trace_name}<br>"
#                             f"Breakpoint {i_bp}<br>"
#                             f"{sec_column}: %{{x}}<br>"
#                             f"{vp_column}: %{{y}}<extra></extra>"
#                         )
#                     )
#                 )

#                 # Línea horizontal opcional
#                 fig.add_hline(
#                     y=bp_vp,
#                     line=dict(color=color, dash='dot', width=1),
#                     opacity=0.45
#                 )

#     fig.update_layout(
#         title=title,
#         xaxis_title=sec_column,
#         yaxis_title=vp_column,
#         template='plotly_white',
#         legend_title='ID',
#         hovermode='closest',
#         height=600
#     )

#     fig.update_yaxes(autorange='reversed')

#     return fig

In [ ]:
import os
import glob
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import sys

root = os.path.abspath('../..')
sys.path.append(root)


def plot_dgh_traces_with_breakpoints_fast(
    path_dgh,
    path_sec_files,
    save_html=False,
    output_dir=None,
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]',
    id_column='ID',
    bp1_sec_column='breakpoint_1 (sec)',
    bp2_sec_column='breakpoint_2 (sec)',
    bp1_vp_column='breakpoint_1 (vp)',
    bp2_vp_column='breakpoint_2 (vp)',
    title='Traces with breakpoints'
):
    """
    Versión optimizada para Plotly:
    - Usa Scattergl (WebGL) para mayor velocidad
    - Hover mínimo: solo x e y
    - Puede guardar a HTML si save_html=True
    - El nombre del HTML se toma del archivo CSV de breakpoints
    - Sin interpolación de breakpoints

    Parámetros
    ----------
    path_dgh : str
        Ruta al CSV con los breakpoints.
    path_sec_files : str
        Carpeta donde están los CSV de traces.
    save_html : bool
        Si True, guarda la figura en HTML.
    output_dir : str or None
        Directorio donde guardar el HTML. Obligatorio si save_html=True.
    """

    # -------------------------
    # Validaciones de guardado
    # -------------------------
    if save_html:
        if output_dir is None:
            raise ValueError("Si save_html=True, debes proporcionar output_dir.")
        os.makedirs(output_dir, exist_ok=True)

    # -------------------------
    # Leer y validar archivo DGH
    # -------------------------
    df_dgh = pd.read_csv(path_dgh)

    required_dgh_cols = [
        id_column,
        bp1_sec_column, bp2_sec_column,
        bp1_vp_column, bp2_vp_column
    ]
    missing_dgh = [c for c in required_dgh_cols if c not in df_dgh.columns]
    if missing_dgh:
        raise ValueError(f"Faltan columnas en path_dgh: {missing_dgh}")

    df_dgh = df_dgh[required_dgh_cols].copy()
    df_dgh[id_column] = df_dgh[id_column].astype(str)

    for col in [bp1_sec_column, bp2_sec_column, bp1_vp_column, bp2_vp_column]:
        df_dgh[col] = pd.to_numeric(df_dgh[col], errors='coerce')

    # -------------------------
    # Colores
    # -------------------------
    unique_ids = df_dgh[id_column].dropna().unique().tolist()
    palette = px.colors.qualitative.Alphabet
    color_map = {id_val: palette[i % len(palette)] for i, id_val in enumerate(unique_ids)}

    fig = go.Figure()
    shown_legend_ids = set()

    # -------------------------
    # Iterar por cada ID
    # -------------------------
    for _, row in df_dgh.iterrows():
        id_val = str(row[id_column])
        color = color_map[id_val]

        breakpoints = [
            (row[bp1_sec_column], row[bp1_vp_column], 1),
            (row[bp2_sec_column], row[bp2_vp_column], 2),
        ]

        pattern = os.path.join(path_sec_files, f"{id_val}*.csv")
        matched_files = sorted(glob.glob(pattern))

        if not matched_files:
            print(f"[AVISO] No se encontraron archivos para ID={id_val} con patrón: {pattern}")
            continue

        for file_path in matched_files:
            try:
                df_trace = pd.read_csv(file_path, usecols=[vp_column, sec_column])
            except Exception as e:
                print(f"[AVISO] No se pudo leer {file_path}: {e}")
                continue

            df_trace[vp_column] = pd.to_numeric(df_trace[vp_column], errors='coerce')
            df_trace[sec_column] = pd.to_numeric(df_trace[sec_column], errors='coerce')
            df_trace = df_trace.dropna(subset=[vp_column, sec_column])

            if df_trace.empty:
                print(f"[AVISO] {os.path.basename(file_path)} quedó vacío después de limpiar NaNs.")
                continue

            x = df_trace[sec_column].to_numpy(dtype=np.float32)
            y = df_trace[vp_column].to_numpy(dtype=np.float32)

            showlegend_line = id_val not in shown_legend_ids

            # Trace principal con WebGL
            fig.add_trace(
                go.Scattergl(
                    x=x,
                    y=y,
                    mode='markers',
                    name=id_val,
                    legendgroup=id_val,
                    showlegend=showlegend_line,
                    marker=dict(
                        color=color,
                        size=3
                    ),
                    hovertemplate='x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>'
                )
            )

            shown_legend_ids.add(id_val)

            # Breakpoints
            bp_x = []
            bp_y = []

            for bp_sec, bp_vp, i_bp in breakpoints:
                if pd.isna(bp_sec) or pd.isna(bp_vp):
                    continue

                bp_x.append(bp_sec)
                bp_y.append(bp_vp)

                # Línea horizontal opcional
                fig.add_shape(
                    type='line',
                    x0=float(np.min(x)),
                    x1=float(np.max(x)),
                    y0=float(bp_vp),
                    y1=float(bp_vp),
                    line=dict(color=color, dash='dot', width=1),
                    opacity=0.35
                )

            if bp_x:
                fig.add_trace(
                    go.Scattergl(
                        x=np.array(bp_x, dtype=np.float32),
                        y=np.array(bp_y, dtype=np.float32),
                        mode='markers',
                        legendgroup=id_val,
                        showlegend=False,
                        marker=dict(
                            color=color,
                            size=10,
                            symbol='x'
                        ),
                        hovertemplate='x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>'
                    )
                )

    fig.update_layout(
        title=title,
        xaxis_title=sec_column,
        yaxis_title=vp_column,
        template='plotly_white',
        legend_title='ID',
        hovermode='closest',
        height=600
    )

    fig.update_yaxes(autorange='reversed')

    # -------------------------
    # Guardar HTML
    # -------------------------
    if save_html:
        base_name = os.path.splitext(os.path.basename(path_dgh))[0]
        output_html = os.path.join(output_dir, f"{base_name}.html")

        fig.write_html(
            output_html,
            include_plotlyjs='cdn',
            full_html=True
        )
        print(f"[OK] HTML guardado en: {output_html}")

    return fig

In [ ]:
def plot_traces_vs_time_fast(
    path_sec_files,
    save_html=False,
    output_dir=None,
    file_pattern="*.csv",
    vp_column='Vertical Position m',
    time_hms_column='Time (HH:mm:ss)',
    time_frac_column='Time (Fract. Sec)',
    output_time_column='time_seconds',
    title='Vertical Position vs Time'
):
    """
    Plot Vertical Position as a function of cumulative elapsed time.

    Logic
    -----
    - Read HH:mm:ss and optionally fractional seconds.
    - Convert HH:mm:ss to seconds explicitly.
    - If fractional seconds exist, use them as sub-second ordering and timing.
    - Sort by:
        1) HH:mm:ss in seconds
        2) fractional seconds (ascending), if available
    - Build an absolute time:
        absolute_time = hms_seconds + fractional_seconds
    - Handle midnight crossing if needed.
    - Build cumulative elapsed time:
        time_seconds = absolute_time - first_absolute_time
    - Print the first 10 rows for debugging.
    - Plot Vertical Position vs time_seconds.
    - Reverse y-axis.
    """

    if save_html:
        if output_dir is None:
            raise ValueError("If save_html=True, you must provide output_dir.")
        os.makedirs(output_dir, exist_ok=True)

    pattern = os.path.join(path_sec_files, file_pattern)
    matched_files = sorted(glob.glob(pattern))

    if not matched_files:
        raise FileNotFoundError(f"No CSV files were found with pattern: {pattern}")

    palette = px.colors.qualitative.Alphabet
    color_map = {
        os.path.splitext(os.path.basename(file_path))[0]: palette[i % len(palette)]
        for i, file_path in enumerate(matched_files)
    }

    fig = go.Figure()

    for file_path in matched_files:
        trace_name = os.path.splitext(os.path.basename(file_path))[0]
        color = color_map[trace_name]

        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"[WARNING] Could not read {file_path}: {e}")
            continue

        required_cols = [vp_column, time_hms_column]
        missing_required = [c for c in required_cols if c not in df.columns]
        if missing_required:
            print(f"[WARNING] Missing required columns in {file_path}: {missing_required}")
            continue

        has_fractional_seconds = time_frac_column in df.columns

        # -------------------------
        # Basic cleaning
        # -------------------------
        df[vp_column] = pd.to_numeric(df[vp_column], errors='coerce')
        df[time_hms_column] = df[time_hms_column].astype(str).str.strip()

        if has_fractional_seconds:
            df[time_frac_column] = pd.to_numeric(df[time_frac_column], errors='coerce')
            df = df.dropna(subset=[vp_column, time_hms_column, time_frac_column]).copy()
        else:
            df = df.dropna(subset=[vp_column, time_hms_column]).copy()

        if df.empty:
            print(f"[WARNING] {os.path.basename(file_path)} is empty after cleaning.")
            continue

        # -------------------------
        # Parse HH:mm:ss explicitly
        # -------------------------
        time_parts = df[time_hms_column].str.split(":", expand=True)

        if time_parts.shape[1] != 3:
            print(f"[WARNING] Invalid HH:mm:ss format in {file_path}.")
            continue

        df['_hour'] = pd.to_numeric(time_parts[0], errors='coerce')
        df['_minute'] = pd.to_numeric(time_parts[1], errors='coerce')
        df['_second'] = pd.to_numeric(time_parts[2], errors='coerce')

        df = df.dropna(subset=['_hour', '_minute', '_second']).copy()

        if df.empty:
            print(f"[WARNING] No valid HH:mm:ss values found in {os.path.basename(file_path)}.")
            continue

        # -------------------------
        # Convert HH:mm:ss to seconds of day
        # -------------------------
        df['_hms_seconds'] = (
            df['_hour'] * 3600
            + df['_minute'] * 60
            + df['_second']
        ).astype(float)

        # -------------------------
        # Fractional seconds
        # -------------------------
        if has_fractional_seconds:
            df['_frac_seconds'] = df[time_frac_column].astype(float)
        else:
            df['_frac_seconds'] = 0.0

        # -------------------------
        # Sort correctly:
        # first by HH:mm:ss, then by fractional seconds
        # -------------------------
        df = df.sort_values(
            by=['_hms_seconds', '_frac_seconds'],
            ascending=[True, True]
        ).reset_index(drop=True)

        # -------------------------
        # Absolute time within day
        # -------------------------
        df['_absolute_time_raw'] = df['_hms_seconds'] + df['_frac_seconds']

        # -------------------------
        # Unwrap midnight crossing if needed
        # -------------------------
        abs_raw = df['_absolute_time_raw'].to_numpy(dtype=float)
        abs_unwrapped = abs_raw.copy()

        day_offset = 0.0
        for i in range(1, len(abs_unwrapped)):
            current_raw = abs_raw[i]
            prev_unwrapped = abs_unwrapped[i - 1]

            current_candidate = current_raw + day_offset

            if current_candidate < prev_unwrapped:
                day_offset += 24 * 3600
                current_candidate = current_raw + day_offset

            abs_unwrapped[i] = current_candidate

        df['_absolute_time_unwrapped'] = abs_unwrapped

        # -------------------------
        # Final cumulative elapsed time
        # -------------------------
        t0 = df['_absolute_time_unwrapped'].iloc[0]
        df[output_time_column] = df['_absolute_time_unwrapped'] - t0

        # -------------------------
        # Optional diagnostic delta
        # -------------------------
        df['_delta_t'] = df[output_time_column].diff()
        if not df.empty:
            df.loc[df.index[0], '_delta_t'] = 0.0

        # -------------------------
        # Debug print
        # -------------------------
        debug_cols = [time_hms_column]
        if has_fractional_seconds:
            debug_cols.append(time_frac_column)
        debug_cols += [
            '_hms_seconds',
            '_frac_seconds',
            '_absolute_time_unwrapped',
            output_time_column,
            '_delta_t'
        ]

        print(f"\n[DEBUG] Profile: {trace_name}")
        print(f"[DEBUG] First 10 rows after sorting and cumulative time construction:")
        print(df[debug_cols].head(10).to_string(index=False))
        print(f"[DEBUG] time range: min={df[output_time_column].min():.3f} s, max={df[output_time_column].max():.3f} s")

        # -------------------------
        # Arrays for plot
        # -------------------------
        x = df[output_time_column].to_numpy(dtype=np.float32)
        y = df[vp_column].to_numpy(dtype=np.float32)

        hover_template = (
            'time=%{x:.3f} s<br>VP=%{y:.3f}<extra></extra>'
            if has_fractional_seconds
            else 'time=%{x:.0f} s<br>VP=%{y:.3f}<extra></extra>'
        )

        fig.add_trace(
            go.Scattergl(
                x=x,
                y=y,
                mode='markers',
                name=trace_name,
                marker=dict(
                    color=color,
                    size=3
                ),
                hovertemplate=hover_template
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title='Time (s)',
        yaxis_title=vp_column,
        template='plotly_white',
        legend_title='Trace',
        hovermode='closest',
        height=600
    )

    fig.update_yaxes(autorange='reversed')

    if save_html:
        folder_name = os.path.basename(os.path.normpath(path_sec_files))
        output_html = os.path.join(output_dir, f"{folder_name}_vs_time.html")

        fig.write_html(
            output_html,
            include_plotlyjs='cdn',
            full_html=True
        )
        print(f"[OK] HTML saved to: {output_html}")

    return fig

# SEC

In [ ]:
path_dgh = f'{root}/results/dgh_breakpoints/dgh_27800_raw_sat51w2p_secuniform_lrs70.csv'
output_dir = f'{root}/results/html_breakpoints'
path_sec_files = f'{root}/data/rawdy/rawdy_sat51w2p_secuniform_lrs70'

fig = plot_dgh_traces_with_breakpoints_fast(
    path_dgh=path_dgh,
    path_sec_files=path_sec_files,
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]',
    save_html=True,
    output_dir=output_dir,
    title='SEC LRS70'
)

fig.show()


# TEMP

In [ ]:
path_dgh = f'{root}/results/dgh_breakpoints/dgh_27800_raw_sat51w2p_lrs70.csv'
output_dir = f'{root}/results/html_breakpoints'
path_sec_files = f'{root}/data/rawdy/rawdy_sat51w2p_lrs70'

fig = plot_dgh_traces_with_breakpoints_fast(
    path_dgh=path_dgh,
    path_sec_files=path_sec_files,
    vp_column='Vertical Position m',
    sec_column='Temp °C',
    save_html=True,
    output_dir=output_dir,
    title='Temperature SEC LRS70'
)

fig.show()

# Time

In [ ]:
output_dir = f'{root}/results/html_time'
path_sec_files = f'{root}/data/raw/raw_lrs69'

fig = plot_traces_vs_time_fast(
    path_sec_files,
    save_html=True,
    output_dir=output_dir,
    file_pattern="*.csv",
    vp_column='Vertical Position m',
    time_hms_column='Time (HH:mm:ss)',
    time_frac_column='Time (Fract. Sec)',
    title='Vertical Position vs Time'
    )

fig.show()